# Procesamiento de PCAP → CSV para el Pipeline de ML IoT

## Propósito de este notebook

Este notebook documenta el proceso completo para convertir los archivos PCAP originales del dataset UNSW IoT Traffic Traces en archivos CSV con etiquetas de clase, siguiendo el enfoque del repositorio **IIsy** (Xiong & Zilberman, 2019).

> **Nota**: Este proceso produce archivos en el formato del script `IIsy/trace_processing/set_features.sh` (12 columnas, sin cabecera, etiqueta como número). El notebook principal `ML_Pipeline_DT.ipynb` usa en cambio los CSV pre-procesados del sitio UNSW (10 columnas, con cabecera), que son el equivalente ya listo — **no es necesario ejecutar este notebook** a menos que quieras verificar el proceso o reproducir el pipeline original de IIsy.

---

## Requisitos

Este notebook está diseñado para ejecutarse en la **VM Ubuntu del laboratorio P4**, donde están disponibles:
- `tshark` (Wireshark CLI)
- `python3` con `pandas`, `numpy`
- Herramientas de descompresión: `tar`, `gzip`

## Archivos de entrada

Los archivos PCAP están comprimidos como `.tar.gz` en:
```
LabML/Legacy/iottraces_dataset/pcap/
  16-09-23.tar.gz
  16-09-24.tar.gz
  ...
  16-10-12.tar.gz
```

Cada archivo `.tar.gz` contiene **uno o más archivos `.pcap`** con el tráfico capturado ese día.

---
## Paso 1: ¿Procesamiento manual necesario?

### Respuesta corta: **Sí**, pero solo en la VM Ubuntu

Los `.tar.gz` **no pueden ser procesados directamente** por `tshark` — deben extraerse primero. Los pasos son:

### 1.1 Extracción de los archivos tar.gz

```bash
# En la VM Ubuntu, desde el directorio de PCAPs:
cd LabML/Legacy/iottraces_dataset/pcap/

# Crear directorio de trabajo
mkdir -p extracted/

# Extraer todos los tar.gz (puede tardar varios minutos)
for f in *.tar.gz; do
    echo "Extrayendo $f..."
    tar -xzf "$f" -C extracted/
done

# Ver qué hay dentro
ls extracted/
```

### 1.2 Estructura esperada dentro de los tar.gz

Cada archivo contiene uno o más `.pcap` organizados por dispositivo o por ventana temporal. El resultado típico es:

```
extracted/
  16-09-23/
    device1.pcap
    device2.pcap
    ...
```

> **Tamaño**: Los PCAP sin comprimir pueden ocupar varios GB por día. Asegúrate de tener espacio suficiente en la VM antes de extraer todos los archivos.

In [ ]:
# =============================================================================
# PASO 1.3: INSPECCIONAR CONTENIDO DE UN TAR.GZ (sin extraer)
# =============================================================================
# Ejecutar en la VM Ubuntu

import subprocess
import os

PCAP_DIR = os.path.join('..', '..', 'Legacy', 'iottraces_dataset', 'pcap')
first_tgz = os.path.join(PCAP_DIR, '16-09-23.tar.gz')

# Listar contenido del primer tar.gz sin extraer
result = subprocess.run(
    ['tar', '-tzf', first_tgz],
    capture_output=True, text=True
)

print(f"Contenido de 16-09-23.tar.gz:")
print(result.stdout[:2000])
if result.returncode != 0:
    print(f"Error: {result.stderr}")
    print("(Este paso requiere ejecutarse en Linux/macOS con 'tar' disponible)")

---
## Paso 2: Extracción de Features con tshark

### El pipeline de IIsy (set_features.sh)

El script original de IIsy (`trace_processing/set_features.sh`) usa `tshark` para extraer **12 campos de cabecera** de cada paquete:

```bash
tshark -r archivo.pcap -Tfields \
    -E occurrence=f -E separator=, \
    -e frame.len    \
    -e eth.type     \
    -e ip.proto     \
    -e ip.flags     \
    -e ipv6.nxt     \
    -e ipv6.opt     \
    -e tcp.srcport  \
    -e tcp.dstport  \
    -e tcp.flags    \
    -e udp.srcport  \
    -e udp.dstport  \
    -e eth.src
```

### Diferencia con el formato del sitio UNSW

| Aspecto | CSV del sitio UNSW | CSV generado por tshark (IIsy) |
|---|---|---|
| Cabecera | Sí (`Packet ID, TIME, ...`) | No (columnas posicionales) |
| Columnas | 10 (puertos TCP/UDP unificados) | 12 (TCP y UDP separados) |
| Tamaño | Más compacto | Más verbose |
| Formato port | `port.src` único | `tcp.srcport` y `udp.srcport` por separado |

Para el ejercicio DT con 3 features, ambos formatos son equivalentes — el sitio UNSW ya unificó los puertos TCP/UDP en `port.src`/`port.dst`.

In [ ]:
# =============================================================================
# PASO 2: EXTRACCIÓN CON TSHARK
# =============================================================================
# IMPORTANTE: Este bloque SOLO funciona en Linux/Ubuntu con tshark instalado.
# En Windows, este proceso debe ejecutarse en la VM del laboratorio.
#
# Verifica que tshark esté disponible:

import subprocess

check = subprocess.run(['which', 'tshark'], capture_output=True, text=True)
if check.returncode == 0:
    print(f"tshark encontrado en: {check.stdout.strip()}")
    ver = subprocess.run(['tshark', '--version'], capture_output=True, text=True)
    print(ver.stdout.split('\n')[0])
else:
    print("tshark NO encontrado.")
    print("En Ubuntu/P4 VM:  sudo apt-get install tshark")
    print("Este notebook debe ejecutarse en la VM del laboratorio.")

In [ ]:
# =============================================================================
# PASO 2.1: PROCESAR UN PCAP CON TSHARK (ejemplo con un solo archivo)
# =============================================================================
# Ajusta estas rutas según donde estén los PCAP extraídos en tu VM

import subprocess, os

# Directorio donde extrajiste los PCAP
EXTRACTED_DIR = '/tmp/iotpcap/extracted/'   # <-- ajustar
OUTPUT_DIR    = '/tmp/iotpcap/tshark_csv/'  # <-- ajustar

os.makedirs(OUTPUT_DIR, exist_ok=True)

TSHARK_FIELDS = [
    'frame.len', 'eth.type', 'ip.proto', 'ip.flags',
    'ipv6.nxt',  'ipv6.opt',
    'tcp.srcport', 'tcp.dstport', 'tcp.flags',
    'udp.srcport', 'udp.dstport',
    'eth.src'
]


def pcap_to_csv_tshark(pcap_path, out_csv_path):
    """
    Convierte un archivo PCAP a CSV usando tshark.
    El CSV tendrá 12 columnas, sin cabecera, separado por comas.
    Los valores vacíos (campos no presentes en el paquete) quedan en blanco.
    """
    cmd = ['tshark', '-r', pcap_path,
           '-Tfields', '-E', 'occurrence=f', '-E', 'separator=,']
    for field in TSHARK_FIELDS:
        cmd += ['-e', field]

    with open(out_csv_path, 'w') as fout:
        result = subprocess.run(cmd, stdout=fout, stderr=subprocess.PIPE, text=True)

    if result.returncode != 0:
        print(f"Error tshark: {result.stderr[:200]}")
        return 0

    # Contar líneas
    with open(out_csv_path) as f:
        n = sum(1 for _ in f)
    return n


# Ejemplo: procesar el primer PCAP encontrado
example_pcaps = []
for root, dirs, files in os.walk(EXTRACTED_DIR):
    for fn in files:
        if fn.endswith('.pcap') or fn.endswith('.pcapng'):
            example_pcaps.append(os.path.join(root, fn))

if example_pcaps:
    pcap = example_pcaps[0]
    out  = os.path.join(OUTPUT_DIR, os.path.basename(pcap).replace('.pcap','.csv'))
    n = pcap_to_csv_tshark(pcap, out)
    print(f"Procesado: {pcap}")
    print(f"Salida:    {out} ({n:,} paquetes)")
else:
    print(f"No se encontraron PCAP en {EXTRACTED_DIR}")
    print("Extrae primero los tar.gz con: tar -xzf *.tar.gz -C extracted/")

---
## Paso 3: Asignación de Etiquetas por MAC

La última columna del CSV generado por tshark contiene la **dirección MAC** del dispositivo fuente (`eth.src`). El script de IIsy sustituye cada MAC por el número de clase correspondiente, usando el archivo `replacement_numeric`.

El archivo `replacement_numeric` tiene el formato:
```
ec:1a:59:7A:02:C5/0    ← MAC/clase
70:88:6b:10:0f:c6/1
...
```

Dispositivos cuya MAC no aparece en el archivo quedan en **clase 4 (Otros)**.

### Comparación con el script bash original de IIsy

El script original `set_features.sh` hace esto con `grep` + `sed` en bash, que es lento para archivos grandes. La implementación Python a continuación hace lo mismo de forma más eficiente.

In [ ]:
# =============================================================================
# PASO 3: ASIGNACIÓN DE ETIQUETAS (Python — equivalente al set_features.sh)
# =============================================================================

import pandas as pd
import numpy as np

# Columnas del CSV generado por tshark (sin cabecera)
TSHARK_COLS = [
    'frame_len', 'eth_type', 'ip_proto', 'ip_flags',
    'ipv6_nxt',  'ipv6_opt',
    'tcp_srcport', 'tcp_dstport', 'tcp_flags',
    'udp_srcport', 'udp_dstport',
    'eth_src'
]

# Mapeo MAC → clase (de IIsy/trace_processing/replacement_numeric)
MAC_TO_CLASS = {
    'ec:1a:59:7a:02:c5': 0, 'ec:1a:59:83:28:11': 0, 'ec:1a:59:79:f4:89': 0,
    'd0:73:d5:01:83:08': 0, '00:24:e4:44:68:44': 0, '74:6a:89:00:2e:25': 0,
    '74:c6:3b:29:d7:1d': 0, '84:f3:eb:52:42:db': 0,
    '70:88:6b:10:0f:c6': 1, '70:ee:50:03:b8:ac': 1,
    '00:24:e4:20:28:c6': 1, '18:b4:30:25:be:e4': 1,
    'f4:f5:d8:d4:eb:12': 2, '44:65:0d:56:cc:d3': 2,
    '18:b7:9e:02:20:44': 2, '28:c2:dd:ff:a5:2d': 2,
    '00:16:6c:ab:6b:88': 3, '88:4a:ea:31:66:9d': 3, '70:ee:50:18:34:43': 3,
    '7c:70:bc:5d:5e:dc': 3, '30:8c:fb:2f:e4:b2': 3, 'b4:75:0e:ec:e5:a9': 3,
    'f4:f2:6d:93:51:f1': 3, 'e0:76:d0:3f:00:ae': 3, 'f4:f5:d8:8f:0a:3c': 3,
}


def load_tshark_csv(csv_path):
    """
    Carga un CSV generado por tshark (sin cabecera, 12 columnas) y
    añade la etiqueta de clase a partir de la MAC.

    Para el ejercicio DT (3 features):
      - ip_proto: columna 'ip_proto'
      - src_port: tcp_srcport si TCP, udp_srcport si UDP
      - dst_port: tcp_dstport si TCP, udp_dstport si UDP
      - label: clase asignada por MAC (4 = Otros)
    """
    df = pd.read_csv(
        csv_path,
        names=TSHARK_COLS,
        na_values=['', ' '],
        low_memory=False
    )

    # Asignar clase desde MAC
    df['label'] = df['eth_src'].str.lower().map(MAC_TO_CLASS).fillna(4).astype('uint8')

    # Unificar puertos: usar TCP si existe, sino UDP, sino 0
    df['src_port'] = df['tcp_srcport'].fillna(df['udp_srcport']).fillna(0).astype('uint16')
    df['dst_port'] = df['tcp_dstport'].fillna(df['udp_dstport']).fillna(0).astype('uint16')

    # Eliminar paquetes sin protocolo IP
    df = df.dropna(subset=['ip_proto'])
    df['ip_proto'] = df['ip_proto'].astype('uint8')

    return df[['ip_proto', 'src_port', 'dst_port', 'label']]


# Prueba con el CSV generado en el Paso 2
import glob

csv_files = glob.glob(os.path.join(OUTPUT_DIR, '*.csv'))
if csv_files:
    df_test = load_tshark_csv(csv_files[0])
    print(f"CSV cargado: {csv_files[0]}")
    print(f"Paquetes con IP: {len(df_test):,}")
    print(f"\nDistribución de clases:")
    class_names = ['Smart-Static', 'Sensores', 'Audio', 'Video', 'Otros']
    for cls, name in enumerate(class_names):
        cnt = (df_test['label'] == cls).sum()
        print(f"  Clase {cls} ({name}): {cnt:,}")
    print(f"\nPrimeras filas:")
    display(df_test.head(10))
else:
    print("No hay archivos CSV en OUTPUT_DIR. Ejecuta primero el Paso 2.")

---
## Paso 4: Pipeline Completo (todos los días)

El siguiente bloque procesa todos los días en un solo loop:
1. Extrae el tar.gz
2. Ejecuta tshark sobre cada PCAP
3. Asigna etiquetas
4. Concatena en un DataFrame único

> **Tiempo estimado**: 2–6 horas para los 20 días completos, dependiendo del hardware.
> Considera ejecutarlo como script de fondo (`nohup python pcap_pipeline.py &`) en la VM.

In [ ]:
# =============================================================================
# PASO 4: PIPELINE COMPLETO tar.gz → DataFrame
# =============================================================================
# NOTA: Este proceso puede tardar varias horas en la VM.
# Ejecutar solo si se quiere verificar el pipeline IIsy desde cero.

import tarfile, tempfile, time

PCAP_DIR   = os.path.join('..', '..', 'Legacy', 'iottraces_dataset', 'pcap')
tgz_files  = sorted(glob.glob(os.path.join(PCAP_DIR, '*.tar.gz')))

all_frames = []
total_t0 = time.time()

for tgz in tgz_files:
    day = os.path.basename(tgz).replace('.tar.gz', '')
    print(f"\nProcesando día: {day}")
    t0 = time.time()

    # Extraer a directorio temporal
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            with tarfile.open(tgz, 'r:gz') as tar:
                tar.extractall(tmpdir)
        except Exception as e:
            print(f"  Error extrayendo {tgz}: {e}")
            continue

        # Encontrar todos los PCAP dentro del tar
        pcap_files = []
        for root, _, files in os.walk(tmpdir):
            for fn in files:
                if fn.endswith('.pcap') or fn.endswith('.pcapng'):
                    pcap_files.append(os.path.join(root, fn))

        print(f"  PCAP encontrados: {len(pcap_files)}")

        day_frames = []
        for pcap in pcap_files:
            # Crear CSV temporal con tshark
            with tempfile.NamedTemporaryFile(suffix='.csv', delete=False, mode='w') as tmp_csv:
                tmp_csv_path = tmp_csv.name

            n = pcap_to_csv_tshark(pcap, tmp_csv_path)
            if n > 0:
                df_pcap = load_tshark_csv(tmp_csv_path)
                day_frames.append(df_pcap)
            os.unlink(tmp_csv_path)

        if day_frames:
            df_day = pd.concat(day_frames, ignore_index=True)
            all_frames.append(df_day)
            elapsed = time.time() - t0
            print(f"  {len(df_day):,} paquetes IP procesados en {elapsed:.1f}s")

# Resultado final
if all_frames:
    df_pcap_full = pd.concat(all_frames, ignore_index=True)
    total_elapsed = time.time() - total_t0
    print(f"\n{'='*50}")
    print(f"TOTAL: {len(df_pcap_full):,} paquetes en {total_elapsed/60:.1f} minutos")
    print(f"Memoria: {df_pcap_full.memory_usage(deep=True).sum()/1e6:.0f} MB")
else:
    print("No se procesaron datos. Verifica que los tar.gz sean accesibles.")

---
## Paso 5: Comparación entre ambos formatos

Si ya tienes `df_pcap_full` (de este notebook) y `df` (del notebook principal cargado desde CSV UNSW), puedes comparar que son equivalentes:

In [ ]:
# =============================================================================
# PASO 5: VERIFICACIÓN — ambos pipelines deben dar distribuciones similares
# =============================================================================

# Cargar el dataset UNSW CSV (mismo que en ML_Pipeline_DT.ipynb)
import sys, os
sys.path.insert(0, '.')  # Para importar funciones si las tienes en script

CSV_DIR = os.path.join('..', '..', 'Legacy', 'iottraces_dataset', 'csv')

# (Necesita pandas importado y MAC_TO_CLASS definido arriba)

if 'df_pcap_full' in dir() and len(all_frames) > 0:
    print("Comparación de distribuciones de clase:")
    print(f"{'Clase':<6} {'CSV UNSW':>12}  {'PCAP+tshark':>12}")
    print("-" * 35)
    class_names = ['Smart-Static', 'Sensores', 'Audio', 'Video', 'Otros']
    for cls in range(5):
        # Necesitaría df de ML_Pipeline_DT — aquí solo mostramos el PCAP
        pcap_cnt = (df_pcap_full['label'] == cls).sum()
        print(f"  {cls} ({class_names[cls]:<12}):  {'N/A':>10}   {pcap_cnt:>10,}")
    print("\nNota: Los totales difieren porque el CSV UNSW puede incluir")
    print("dispositivos que los PCAP organizan por MAC separadamente.")
else:
    print("Ejecuta primero el Paso 4 para tener df_pcap_full disponible.")

---
## Resumen: PCAP vs CSV UNSW

| Aspecto | Desde PCAP (este notebook) | CSV del sitio UNSW (ML_Pipeline_DT.ipynb) |
|---|---|---|
| Formato entrada | `.tar.gz` con archivos `.pcap` | `.csv.zip` directamente |
| Herramientas | `tshark` + `tar` (Linux) | Solo `pandas` (cualquier OS) |
| Tiempo de proceso | 2–6 horas | 2–5 minutos |
| Datos obtenidos | Mismos (mismo dataset original) | Mismos |
| Uso recomendado | Verificación del pipeline IIsy | Entrenamiento del DT (rápido) |

### Recomendación

Para el ejercicio de laboratorio y para replicar los árboles de referencia, **usa directamente el notebook `ML_Pipeline_DT.ipynb`** con los CSV del sitio UNSW. Este notebook de PCAP sirve para:
- Entender el pipeline completo de IIsy desde cero
- Verificar que los datos son consistentes entre formatos
- Experimentar con features adicionales no presentes en el CSV UNSW (e.g., `ip.flags`, `tcp.flags`)

### Referencias

1. Xiong, Z., & Zilberman, N. (2019). *Do Switches Dream of Machine Learning?* ACM HotNets '19.
2. Sivanathan et al. (2018). *Classifying IoT devices in smart environments.* IEEE TMC.
3. UNSW IoT Traffic Traces: https://iotanalytics.unsw.edu.au/iottraces.html
4. Repositorio IIsy: `LabML/Legacy/IIsy/trace_processing/set_features.sh`